# Práctica 6. Reconocimiento de suelos con y sin descriptores de Haralick

Dataset: **Soil DataSet** (Kaggle, `naivedatamodel/soil-dataset`). Se unen `train` y `test`, se eliminan duplicados exactos y se evalúan dos experimentos con el mismo protocolo:

- **Sin Haralick:** píxeles en escala de grises (64 × 64) → estandarización → PCA (95 % de varianza) → SVM con núcleo RBF.
- **Con Haralick:** descriptores de Haralick de la GLCM de los píxeles, distancias 1, 2 y 4 px → estandarización → SVM con núcleo RBF.

Ejecute las celdas en orden (**Kernel → Restart & Run All**). Abra el notebook desde la carpeta `06-reconocimiento-suelos/`.

## 1. Preparación

Verifica que el notebook esté dentro de la carpeta que contiene `src/`.

In [ ]:
import os
import sys
from pathlib import Path

RAIZ = Path.cwd()
if not (RAIZ / "src").is_dir():
    raise RuntimeError(f"No encuentro src/ en {RAIZ}. Abra el notebook desde la carpeta 06-reconocimiento-suelos.")
sys.path.insert(0, str(RAIZ))

import pandas as pd
from IPython.display import Image, display
from src import datos, experimento

## 2. Dataset

Se descarga con `kagglehub` (requiere credenciales de Kaggle y haber aceptado los términos del dataset en su página). Si ya tiene la carpeta descargada, defina la variable de entorno `SOIL_DATOS` con su ruta antes de iniciar Jupyter.

In [ ]:
ruta = os.environ.get("SOIL_DATOS")
if ruta is None:
    import kagglehub
    ruta = kagglehub.dataset_download("naivedatamodel/soil-dataset")
RUTA_DATOS = Path(ruta)
print("Ruta del dataset:", RUTA_DATOS)

## 3. Catálogo y conteo por clase

In [ ]:
filas = datos.catalogo(RUTA_DATOS)
catalogo = pd.DataFrame(filas, columns=["ruta", "clase"])
display(catalogo["clase"].value_counts().rename("imágenes").to_frame())
print("Total de archivos antes de deduplicar:", len(catalogo))

## 4. Ejecución de los dos experimentos

Tarda unos minutos: 5 semillas × 2 experimentos, con bootstrap (2 000 remuestreos) y prueba de permutación (100 permutaciones).

In [ ]:
SALIDA = RAIZ / "salidas"
rep = experimento.ejecutar(
    RUTA_DATOS, SALIDA,
    lado=128, semillas=[0, 1, 2, 3, 4],
    n_boot=2000, n_perm=100, n_jobs=-1,
)
print("Imágenes finales tras deduplicar:", rep["imagenes_finales"])
print("Duplicados exactos eliminados:", rep["duplicados_exactos"])

## 5. Resultados por experimento

Exactitud media en prueba con intervalo de confianza t al 95 % sobre las 5 semillas.

In [ ]:
resumen = pd.DataFrame([
    {"Experimento": v["nombre"],
     "Exactitud media": round(v["exactitud_media"], 4),
     "IC 95 % inferior": round(v["ic95_t_inf"], 4),
     "IC 95 % superior": round(v["ic95_t_sup"], 4),
     "F1 macro medio": round(v["f1_macro_media"], 4),
     "Dimensión": rep["costo_extraccion"]["dimension"][clave]}
    for clave, v in rep["resumen"].items()
])
display(resumen)

## 6. Comparación pareada

Diferencia (con − sin Haralick), prueba t pareada sobre semillas y prueba de McNemar por semilla.

In [ ]:
comp = rep["comparacion"]
print("Diferencia media (con − sin):", round(comp["diferencia_media_con_menos_sin"], 4),
      "IC 95 %:", [round(v, 4) for v in comp["ic95_diferencia"]])
print("Prueba t pareada:", {k: round(v, 4) for k, v in comp["t_pareada_semillas"].items()})
pd.DataFrame(comp["mcnemar_por_semilla"]).assign(semilla=rep["semillas"])

## 7. Prueba de permutación (control de fuga e inflación)

Si la exactitud observada no supera la distribución nula, no hay evidencia de aprendizaje real.

In [ ]:
pd.DataFrame(rep["permutacion"]).T

## 8. Figuras

In [ ]:
for nombre in ["muestra_por_clase.png", "exactitud_ic95.png",
               "confusion_sin_haralick.png", "confusion_con_haralick.png",
               "permutacion_sin_haralick.png", "permutacion_con_haralick.png"]:
    display(Image(filename=str(SALIDA / "figuras" / nombre)))

## 9. F1 por clase

In [ ]:
pd.read_csv(SALIDA / "f1_por_clase.csv").round(4)